In [ ]:
import warnings
from pathlib import Path
import matplotlib.pyplot as plt

import prism

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericMaterials,
    GenericStocks,
    MaterialIntensities,
    SharesInflowStocks,
    ElectricVehicleBatteries,
    EvBatteryLinkModule
)
from imagematerials.preprocessing import get_preprocessing_data

from imagematerials.electricity.constants import STANDARD_SCEN_EXTERNAL_DATA

path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_data = Path(path_base, "data", "raw")

In [ ]:
scenario_list = {"SSP2_baseline":("SSP2_baseline", None),
                # "SSP2_narrow":("SSP2_narrow", "narrow"),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 2000
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_data, "image", climate_scen)
    # climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    vhc_sector = get_preprocessing_data("vehicles", path_data, 
                                    climate_policy_scenario_dir = climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
    ev_battery_sector = get_preprocessing_data("ev_battery", path_data,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario)     
    elc_sector = get_preprocessing_data("electricity", path_data,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario) 
    # elc_sector is a list of preprocessing data for each electricity subsector
    

    factory = ModelFactory(
        [vhc_sector, ev_battery_sector, *elc_sector], complete_timeline
        ).add(GenericStocks, ["vehicles",
                              "elc_gen",
                              "elc_grid_lines",
                              "elc_grid_add",
                              "elc_stor_phs"]
        # ).add(GenericMaterials, ["vehicles"]
        ).add(ElectricVehicleBatteries, ["ev_battery"], input_sources={
        "stock_by_cohort": "vehicles",
        "inflow": "vehicles",
        "outflow_by_cohort": "vehicles"}
        ).add(EvBatteryLinkModule, ["elc_stor_other"], input_sources={
        "stock_battery_kWh_v2g": "ev_battery"}
        ).add(SharesInflowStocks, ["elc_stor_other"],
        # ).add(MaterialIntensities, ["elc_gen",
        #                             "elc_grid_lines",
        #                             "elc_grid_add",
        #                             "elc_stor_phs",
        #                             "elc_stor_other"
        #                             ]
        )
    model = factory.finish()

    model.simulate(simulation_timeline)
    all_output[scen_id] = model
    print(f"Finished {scen_id}")

list(model.ev_battery)

In [ ]:
# Plot: 1 variable over time

model = all_output["SSP2_baseline"] # change to the scenario you want to plot
var = "inflow_battery_kWh" # change to the variable you want to plot
m = model.ev_battery[var].to_array().sum(["Region", "BatteryType"])
unit = prism.U_(m) # get the unit of the variable saved in the dataarray

fig, ax = plt.subplots(figsize=(8,6))
for t in m.Type.values:
    plt.plot(m.time, m.sel(Type=t), label=str(t))

plt.grid(alpha=0.4)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlabel("Year", fontsize=14)
plt.ylabel(f"{var} ({unit})", fontsize=14)
plt.legend(bbox_to_anchor=(1.02, 0.5), loc="center left")
plt.title(f"{var}", fontsize=14)